# A Classification Tree on Real Data

**Notebook 2 of 5.** The six-point tree grew until every leaf was pure and scored 100% on
its own training data. Here is the same thing on 45,222 real people, and three ways to stop
it.

The task: **predict whether a person earns more than $50,000 a year** from a census
record.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, roc_auc_score

## 1. The data

One row per person, thirteen columns describing them, and `income` saying which side of
$50,000 they fall on.

In [ ]:
df = pd.read_csv("data/census.csv")
print(df.shape)
df.head()

In [ ]:
df["income"].value_counts(normalize=True)

About **one person in four** earns more than $50,000 — so a model predicting `<=50K` for
everybody is already right 75% of the time. That is the bar to beat.

The target becomes 1 for `>50K` and 0 otherwise.

In [ ]:
# 1 for the class we are predicting, 0 for the other: scikit-learn wants numbers
y = (df["income"] == ">50K").astype(int)
y.mean().round(4)

## 2. Categorical columns have to become numbers

A scikit-learn tree asks `feature <= threshold`. No threshold makes sense for the string
`"Bachelors"`, so these columns cannot go in as they are.

In [ ]:
# the columns holding strings; every other column is already numeric
categorical = [
    "workclass", "education_level", "marital-status", "occupation",
    "relationship", "race", "sex", "native-country",
]

df[categorical].nunique()

Two ways to turn a category into a number, and only one of them is safe:

- **one-hot encoding** gives each category its own 0/1 column, which a threshold can handle:
  `occupation_Exec-managerial <= 0.5` means *is this person not an executive*;
- **numbering** the categories 0, 1, 2, ... invents an order the data does not have, letting
  the tree ask *is occupation below 7*, which means nothing.

> **Note.** Some gradient boosting libraries take categorical columns directly, and handle
> them better than one-hot encoding does. A scikit-learn tree does not.

In [ ]:
# one 0/1 column per category, so a threshold question can be asked of each
X = pd.get_dummies(df.drop(columns=["income"]), columns=categorical)

print(f"{df.shape[1] - 1} columns became {X.shape[1]}")
X.head()

## 3. Split, then a baseline

Stratifying keeps the one-in-four ratio in both halves.

In [ ]:
# stratify keeps the same one-in-four ratio of >50K in both halves
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# predict <=50K for everyone: the accuracy is just the share that really is <=50K
baseline_accuracy = (y_test == 0).mean()
print(f"{len(X_train)} training rows, {len(X_test)} test rows")
print(f"majority-class baseline accuracy: {baseline_accuracy:.4f}")

## 4. A tree with nothing holding it back

In [ ]:
# no limits at all, so the tree splits until every leaf is pure
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

print(f"depth: {tree.tree_.max_depth}, leaves: {tree.get_n_leaves()}")

**49 questions deep, 6,431 leaves** — about six training rows each. That is not a pattern in
the population; it is the training set carved into thousands of pockets.

> **Exercise:**
>
> Compute this tree's accuracy on the **training** set and the **test** set, and compare both
> with the baseline.

<details><summary>
Click here for a hint...
</summary>

`accuracy_score(y_train, tree.predict(X_train))`, and the same with `y_test` and `X_test`.

</details>

In [ ]:
# your code: training accuracy, test accuracy, and the baseline for comparison

<details><summary>
Click here to compare your answer...
</summary>

**Train 0.9725, test 0.8155** — sixteen points apart, and only 6 points above always
guessing `<=50K`.

That gap is **overfitting**: the tree learned the training rows themselves, noise included,
rather than the pattern behind them. With 6,431 leaves it has room to memorize all of it.

</details>

It is sharper in the ROC AUC, which reads the predicted probabilities rather than the
labels.

In [ ]:
# column 1 of predict_proba is P(>50K), the probability ROC AUC reads
train_auc = roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1])
test_auc = roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1])

print(f"train ROC AUC: {train_auc:.4f}")
print(f"test ROC AUC:  {test_auc:.4f}")

Almost 1.0 on the training set: pure leaves make every probability 0 or 1, and on training
data all of them are right.

Three ways to stop this follow, one section each.

## 5. Stop early: `max_depth`

Refuse to ask more than a set number of questions. One tree per depth, both halves scored,
gives a **validation curve**.

In [ ]:
depths = range(1, 21)
train_scores, test_scores = [], []

# one tree per depth, each scored on both halves: together they are the validation curve
for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    test_scores.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, marker="o", label="train")
plt.plot(depths, test_scores, marker="o", label="test")

# the baseline as a floor, so the curves are read against doing nothing
plt.axhline(baseline_accuracy, color="grey", linestyle="--", label="baseline")
plt.xlabel("max_depth"); plt.ylabel("accuracy")
plt.xticks(list(depths))
plt.legend(); plt.title("Accuracy against tree depth")
plt.show()

In [ ]:
# the depth where the test curve peaks, before overfitting takes over
best_depth = depths[int(np.argmax(test_scores))]
print(f"best depth: {best_depth}, test accuracy {max(test_scores):.4f}")

This is the shape to remember. Both curves climb together while the tree is learning
something real, then part around depth 8 — training keeps rising, test falls. Everything
after the parting is memorization.

**Depth 8 scores 0.8521**, almost four points above the unconstrained tree, with 118 leaves
instead of 6,431.

## 6. Keep leaves big: `min_samples_leaf`

Depth cuts every branch at the same length, whether it was still learning or not.
`min_samples_leaf` limits how *small* a leaf may get instead, so a branch can stay long
where the data supports it.

> **Exercise:**
>
> For each `min_samples_leaf` in `[1, 5, 20, 50, 100]`, print the depth, the leaf count and
> the test accuracy. Which wins, and what happened to the depth?

<details><summary>
Click here for a hint...
</summary>

`DecisionTreeClassifier(min_samples_leaf=value, random_state=42)` in a loop, then
`model.tree_.max_depth`, `model.get_n_leaves()` and `accuracy_score(...)`.

</details>

In [ ]:
# your code: loop over min_samples_leaf values and print depth, leaves and test accuracy

<details><summary>
Click here to compare your answer...
</summary>

**50 wins at 0.8516**, a whisker behind depth 8 — but at depth **22**. That is the point: it
keeps asking wherever a few hundred rows justify it, and stops only where the rows run out.

`min_samples_leaf=1` is the default, which is why section 4's tree had no limit.

</details>

## 7. Grow it, then cut it back: `ccp_alpha`

Both limits above are guesses made **before** the tree is built. **Cost-complexity pruning**
does the opposite: grow it fully, then cut the branches that did not earn their keep. A
pruned tree $T$, with $|T|$ leaves, is scored as

$$R_\alpha(T) = R(T) + \alpha \, |T|$$

where $R(T)$ is the total weighted impurity of those leaves and $\alpha \ge 0$ is the price
you set on each one. Every leaf costs $\alpha$, so a branch survives only if it lowers
$R(T)$ by more than its leaves cost. A larger $\alpha$ prunes harder.

In [ ]:
# alpha 0.0 prunes nothing, so the first row is the unlimited tree again
for alpha in [0.0, 0.0001, 0.0005, 0.002]:
    model = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    model.fit(X_train, y_train)
    print(
        f"ccp_alpha {alpha:<7}  leaves {model.get_n_leaves():5d}"
        f"   train {accuracy_score(y_train, model.predict(X_train)):.4f}"
        f"   test {accuracy_score(y_test, model.predict(X_test)):.4f}"
    )

`ccp_alpha=0.0001` cuts **6,431 leaves to 110** and lifts test accuracy to **0.8537**, the
best of the three — without anyone guessing a depth first.

One honesty note: all of this was compared on the test set, which is what a test set is not
for. Choosing hyperparameters by test score leaks it into the model and flatters the number
you report. The proper tool is cross-validation on the training data, and notebook 05 uses
it. Here the test set is showing the effect, not selecting the model.

## 8. What the pruned tree actually asks

In [ ]:
pruned = DecisionTreeClassifier(ccp_alpha=0.0001, random_state=42)
pruned.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(16, 7))

# max_depth=2 limits the drawing, not the tree: it was fitted in full above
plot_tree(
    pruned, max_depth=2, feature_names=X.columns, class_names=["<=50K", ">50K"],
    filled=True, fontsize=9, ax=ax,
)
plt.show()

The root asks whether the person is **married**, then **capital gains** on one side and
**years of education** on the other. Marriage comes before everything else. Notebook 04 comes back
to reading a fitted tree, and to how much weight that first question really deserves.

## Summary

- Unlimited, the tree reached **depth 49 and 6,431 leaves**: 0.9725 on training data, 0.8155
  on test. That gap is **overfitting**.
- **Categorical columns must be encoded** first; one-hot avoids inventing an order.
- Three ways to control growth: **`max_depth`** (8 → 0.8521), **`min_samples_leaf`**
  (50 → 0.8516), **`ccp_alpha`** (1e-4 → 110 leaves, 0.8537, the best).
- A **validation curve** makes the choice visible: where the curves part, learning stops and
  memorizing starts.

Next, in `03_regression_tree.ipynb`: the same algorithm predicting a number, and the one
thing it cannot do.